# 07 — External Reporting & Submission Quality Control
Demonstrates data preparation, validation, and reconciliation for an external reporting entity.

In [1]:
from pathlib import Path
import pandas as pd
import sqlite3

ROOT = Path("..")
submission = pd.read_csv(ROOT / "data/processed/external_submission_validated.csv")

checks = {
    "duplicate_reporting_keys": submission.duplicated(["term_id","degree_level"]).sum(),
    "missing_required_fields": submission[["term_id","degree_level","enrolled_headcount","fte"]].isna().any(axis=1).sum(),
    "negative_headcount": (submission["enrolled_headcount"] < 0).sum(),
    "negative_fte": (submission["fte"] < 0).sum(),
    "rate_out_of_range": (~submission["credit_completion_rate"].between(0,1)).sum(),
    "failed_row_qc": (submission["qc_pass"] != 1).sum(),
}
pd.Series(checks, name="issues")

duplicate_reporting_keys    0
missing_required_fields     0
negative_headcount          0
negative_fte                0
rate_out_of_range           0
failed_row_qc               0
Name: issues, dtype: int64

In [2]:
db = ROOT / "database/institutional_research.db"
sql = '''
SELECT
    e.term_id,
    s.degree_level,
    COUNT(DISTINCT e.student_id) AS warehouse_headcount
FROM fact_enrollment e
JOIN dim_student s ON e.student_id = s.student_id
GROUP BY e.term_id, s.degree_level
'''
with sqlite3.connect(db) as conn:
    warehouse = pd.read_sql_query(sql, conn)

recon = submission.merge(warehouse, on=["term_id","degree_level"])
recon["difference"] = recon["enrolled_headcount"] - recon["warehouse_headcount"]
recon[["term_id","degree_level","enrolled_headcount","warehouse_headcount","difference"]].head(10)

,term_id,degree_level,enrolled_headcount,warehouse_headcount,difference
0,2022FA,Bachelor's,1276,1276,0
1,2022FA,Master's,242,242,0
2,2023FA,Bachelor's,3849,3849,0
3,2023FA,Master's,769,769,0
4,2023SP,Bachelor's,2349,2349,0
5,2023SP,Master's,475,475,0
6,2023SU,Bachelor's,2958,2958,0
7,2023SU,Master's,612,612,0
8,2024FA,Bachelor's,5775,5775,0
9,2024FA,Master's,1095,1095,0


A file is submission-ready only after required checks pass and aggregate counts reconcile to the authoritative warehouse.